# Note Preliminari

Ciao! Ho inserito varie note in questo file per commentare il lavoro eseguito. 

Come richiesto, qua ho inserito gli step 1 e 3, mentre lo step 2 è nel file addFile.py

***Stefano Nocco***

# Progetto Numpy Start2Impact

## Step1

Nella cartella *files* trovi 8 file:
- 2 file di testo
- 2 file audio
- 4 immagini,

con varie estensioni.

**Step 1**

Inizia creando, in un notebook, uno script Python che iteri in ordine alfabetico sui file della cartella files e, a seconda del tipo (audio, documento, immagine), li sposti nella relativa sottocartella (qui sotto trovi un esempio). Se la sottocartella non esiste, il tuo script dovrà crearla automaticamente.

Durante il ciclo, lo script deve stampare le informazioni dei file: nome, tipo e dimensione in byte. Questo è l'output desiderato:

In [3]:
#FILL ME (OUTPUT MANTENUTO PER CONFRONTO)

bw type:image size:94926B
ciao type:doc size:12B
daffodil type:image size:24657B
eclipse type:image size:64243B
pippo type:doc size:8299B
song1 type:audio size:1087849B
song2 type:audio size:764176B
trump type:image size:10195B


Oltre a stamparne le informazioni via via che li sposti, tieni traccia dei file creando un documento *recap.csv* con le stesse informazioni. Trovi un esempio in questa cartella.

La struttura finale della cartella files dovrà essere:

        - files            
            - audio
                - song1.mp3
                - song2.mp3
            - docs
                - ciao.txt
                - pippo.odt
            - images
                - bw.png
                - daffodil.jpg
                - eclipse.png
                - trump.jpeg    
            - recap.csv

Commenta il codice con i passaggi che fai. Questo vale anche per i prossimi Step.

**Attenzione**: lo script, ogni volta che viene lanciato per spostare nuovi file, deve *aggiornare* (e non sovrascrivere) le sottocartelle e il file di recap. Per controllare che tutto funzioni correttamente, puoi aggiungere altri file alla cartella files e fare un test; oppure, puoi dividere gli 8 file originali in due gruppi e lasciarne uno per il test.

**Consiglio**: puoi usare le librerie *os*, *shutil* e *csv*. 
                
---

In [21]:
from pathlib import Path
import shutil
import csv

# Mapping fra estensioni e categorie di file
FILE_TYPES: dict[str, list[str]] = {
    "images": [".png", ".jpg", ".jpeg", ".gif", ".bmp"],
    "audio": [".mp3", ".wav", ".ogg", ".flac"],
    "docs":   [".txt", ".doc", ".docx", ".pdf", ".odt"]
}


def get_category(ext: str) -> str:
    """
    Restituisce la categoria corrispondente all'estensione del file.
    Se non trova corrispondenza, ritorna 'other'.
    """
    ext = ext.lower()
    for category, extensions in FILE_TYPES.items():
        if ext in extensions:
            return category
    return "other"

def singularize(category: str) -> str:
    """
    Converte un nome di cartella plurale in forma singolare per la stampa del type.
    Es: 'images' -> 'image', 'docs' -> 'doc', 'audio' resta 'audio'.
    """
    return category[:-1] if category.endswith('s') else category


def organize_files(source_dir: Path) -> None:
    """
    Itera i file nella `source_dir`, stampa le info, muove i file 
    nella cartella di appartenenza a aggiunge le loro info al file recap.
    """
    # --- PREPARAZIONE recap.csv ---
    recap_path = source_dir / "recap.csv"
    # Se recap.csv non esiste, crealo e aggiungi l'header
    if not recap_path.exists():
        with open(recap_path, "w", newline="", encoding="utf-8") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(["name", "type", "size(B)"])  # header

    # Carico in memoria le righe già presenti per evitare duplicati nel file recap.csv
    existing_entries: set[tuple[str, str, int]] = set()
    with open(recap_path, "r", newline="", encoding="utf-8") as csvfile:
        reader = csv.reader(csvfile)
        next(reader, None)  # skip header
        for row in reader:
            try:
                name, cat, size_str = row
                existing_entries.add((name, cat, int(size_str)))
            except ValueError:
                # salta righe mal formattate
                continue
            
    # --- ITERAZIONE SUI FILE ---
    for file_path in sorted(source_dir.iterdir(), key=lambda p: p.name.lower()):
        # Salta le cartelle e il file recap.csv
        if not file_path.is_file() or file_path.name == recap_path.name:
            continue

        # Estrai nome, categoria e dimensione
        name = file_path.stem
        category = get_category(file_path.suffix) # Chiamo la funzione definita prima
        size = file_path.stat().st_size
        display_type = singularize(category)

        # Stampa su console (si può usare anche logging)
        print(f"{name} type:{display_type} size:{size}B")

        # Crea la sottocartella se non esiste
        # Non è una divisione, è l'operatore / di pathlib.Path
        # Serve per concatenare le componenti del path, è simile al .join di os
        # é meglio perchè abilita la gestione cross-platform dei separatori (“/” su Linux/macOS, “\” su Windows)
        target_dir = source_dir / category 
        target_dir.mkdir(exist_ok=True)

        # Sposta il file
        dest = target_dir / file_path.name
        shutil.move(str(file_path), str(dest))

        # Registra su recap.csv (append) se e solo se già non sono registrati
        entry = (name, category, size)
        if entry not in existing_entries:
            with open(recap_path, "a", newline="", encoding="utf-8") as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow([name, category, size])
            existing_entries.add(entry)

In [22]:
base_folder = Path("files")  # o il percorso completo
organize_files(base_folder)

bw type:image size:94926B
ciao type:doc size:12B
daffodil type:image size:24657B
eclipse type:image size:64243B
pippo type:doc size:8299B
song1 type:audio size:1087849B
song2 type:audio size:764176B
trump type:image size:10195B


L'Output combacia perfettamente con quanto richiesto nelle specifiche.

# Note sullo Step 1

## Come funziona
Nello Step 1 l'obiettivo è quello di raccogliere tutti i file grezzi nella cartella di lavoro e organizzarli in sottocartelle basate sulla loro estensione, tenendo traccia delle operazioni in un file di riepilogo. All’avvio, lo script verifica se esiste già recap.csv; in caso contrario lo crea e vi scrive l’intestazione name,type,size(B). Dopodiché legge ogni riga già presente, in modo da non duplicare mai la stessa voce nel file di riepilogo.

A questo punto passa in rassegna ogni elemento della cartella principale, ignorando le directory e il medesimo recap.csv. Per ciascun file estrae il nome senza estensione, ne ricava la categoria consultando un dizionario che mappa gruppi di estensioni alle cartelle “images”, “audio” e “docs”, e ne ottiene la dimensione in byte. Prima di spostarlo, scrive a video una riga del tipo nome type:categoria size:numeroB (con “categoria” in forma singolare, grazie a una piccola funzione di conversione). Se la cartella di destinazione non esiste ancora, la crea al volo; poi sposta fisicamente il file al suo interno. Infine, solo se non era già stato annotato in precedenza, aggiunge una riga corrispondente in recap.csv e la marca come già elaborata in memoria.

In questo modo, ogni volta che rilanciamo lo script, i nuovi file vengono classificati e registrati automaticamente senza mai sovrascrivere o duplicare i dati già archiviati.

## File Recap
Nei consigli relativi alla realizzazione del file di Recap c'è scritto (cito il testo):

"
***Le specifiche chiedono di aggiornare il file di recap. Sembra una richiesta semplice, ma molti studenti si perdono qualche pezzo.***

***Prima di buttarti sul codice, prova a chiederti:***

***Com’è fatto il file di recap?***
***Quali operazioni devono essere eseguite se il file non esiste?***
***E se invece esiste già?***
"

In merito a ciò, ho verificato che tale file esistesse nella cartella, con esito negativo. Vi è, infatti, solamente il file recap_esempio.csv, ma in posizione esterna alla cartella files. Pertanto, ho ricreato da zero il file .csv (seguendo le specifiche sopra indicate e seguendo la formattazione del file recap_esempio.csv)

## Funzione singularize(): 
Ho creato questa funzione, perchè le cartelle hanno nome plurale, mentre quanto si stampa a schermo il tipo del file deve essere singolare.
La funzione è molto semplice, funzionale in un ambito cosi ristretto. Potrebbe creare degli errori con nomi di categoria al plurale irregolari.

---

# Step 2 (addfile.py)

**Step 2**

Inserisci lo script che hai creato in un piccolo eseguibile (chiamalo *addfile.py* e posizionalo in questa cartella, a fianco del notebook) dotato di *interfaccia a linea di comando* (CLI).

Lo scopo dell'eseguibile è spostare un *singolo* file (che si trova nella cartella files) nella sottocartella di competenza, aggiornando il recap.

L'interfaccia dell'eseguibile ha come unico argomento (obbligatorio) il nome del file da spostare (comprensivo di formato, es: 'trump.jpeg'). Nel caso in cui il file passato come argomento non esista, l'interfaccia deve comunicarlo all'utente.

**Consiglio**: oltre alle precedenti, puoi usare le librerie *sys* e *argparse*.

---

# Note sullo Step 2

## Come Funziona
In Step 2 ho riadattato e reso il mio script un piccolo tool interattivo da riga di comando. Appena lo avvio, cerco autonomamente la cartella files risalendo i livelli di directory fino a trovarla, così non devo preoccuparmi da dove lo lancio. Poi, se necessario, creo recap.csv con la sua intestazione e carico in memoria tutte le voci già esistenti per evitare duplicati. A questo punto entro in un semplice loop: mostro un prompt, ti invito a digitare il nome di un file presente in files, e – non appena confermi – lo sposto nella sottocartella corretta (creandola al volo se manca), ne stampo a video le informazioni (name, type: categoria, size: byte B) e lo registro in recap.csv solo se non l’avevo già fatto. Quando hai finito di trasferire file, basta che digiti exit, quit o q per uscire dal ciclo e chiudere il programma, conservando intatto tutto lo storico delle operazioni.

P.s. Ovviamente la CLI è cosi per rispondere alle richieste, io implementerei diverse funzioni, in primis una per la visualizzazione dei file presenti all'interno delle cartelle (usando mqagari os.scandir o pathlib.Path).

# Step 3

**Step 3**

Una immagine in scala di grigio ha un solo livello di colore, una RGB ne ha 3, una RGBA 4 (l'ultimo è detto canale *alpha*).

Il modulo *Image* della libreria *PIL* permette di caricare un'immagine, che può essere trasformata in un array NumPy attraverso la funzione *np.array*. A partire da tale array, è possibile capire se l'immagine caricata è in scala di grigio, RGB o RGBA.

Aggiungi al notebook dello Step 1 uno script che iteri sulla sottocartella *images* e costruisca una tabella riassuntiva come questa (prodotta con la libreria *tabulate*):

In [11]:
#FILL ME (OUTPUT MANTENUTO PER CONFRONTO)

╒══════════╤══════════╤═════════╤═════════════╤════════╤════════╤═══════╤═════════╕
│ name     │   height │   width │   grayscale │      R │      G │     B │   ALPHA │
╞══════════╪══════════╪═════════╪═════════════╪════════╪════════╪═══════╪═════════╡
│ bw       │      512 │     512 │       21.48 │   0.00 │   0.00 │  0.00 │    0.00 │
├──────────┼──────────┼─────────┼─────────────┼────────┼────────┼───────┼─────────┤
│ daffodil │      500 │     335 │        0.00 │ 109.25 │  85.56 │  4.97 │    0.00 │
├──────────┼──────────┼─────────┼─────────────┼────────┼────────┼───────┼─────────┤
│ eclipse  │      256 │     256 │        0.00 │ 109.05 │ 109.52 │ 39.85 │  133.59 │
├──────────┼──────────┼─────────┼─────────────┼────────┼────────┼───────┼─────────┤
│ trump    │      183 │     275 │        0.00 │  97.01 │  98.99 │ 90.92 │    0.00 │
╘══════════╧══════════╧═════════╧═════════════╧════════╧════════╧═══════╧═════════╛


Oltre al nome del file, la tabella riporta:

- altezza dell'immagine, in pixel
- larghezza dell'immagine, in pixel
- se l'immagine è in scala di grigio, la colonna *grayscale* indica la media dei valori dell'unico livello di colore
- se l'immagine è a colori, le altre colonne indicano la media dei valori di ogni livello di colore.

---

**Dovrai consegnare**:
- un notebook con gli Step 1 e 3; per semplicità puoi chiamarlo come questo
- addfile.py con quanto richiesto dallo Step 2.

In [28]:
from pathlib import Path
from PIL import Image
import numpy as np
from tabulate import tabulate

images_dir = Path("files/images")
rows: list[list] = []

for img_file in sorted(images_dir.iterdir(), key=lambda p: p.name):
    if not img_file.is_file():
        continue

    name = img_file.stem
    img = Image.open(img_file)
    arr = np.array(img)

    # inizializzo tutte le colonne a 0
    gray_mean   = 0.0
    r_mean      = 0.0
    g_mean      = 0.0
    b_mean      = 0.0
    alpha_mean  = 0.0

    # Scala di grigio
    if arr.ndim == 2:
        height, width = arr.shape
        gray_mean = float(arr.mean())

    # Immagine RGB o RGBA
    elif arr.ndim == 3:
        height, width, channels = arr.shape
        means = arr.mean(axis=(0, 1))
        if channels == 3:
            r_mean, g_mean, b_mean = map(float, means)
        elif channels == 4:
            r_mean, g_mean, b_mean, alpha_mean = map(float, means)

    else:
        # formato non gestito
        continue

    rows.append([name, height, width, gray_mean, r_mean, g_mean, b_mean, alpha_mean])

# Stampo la tabella fancy_grid con due decimali
headers = ["name", "height", "width", "grayscale", "R", "G", "B", "ALPHA"]
print(tabulate(rows, headers=headers, tablefmt="fancy_grid", floatfmt=".2f"))

╒══════════╤══════════╤═════════╤═════════════╤════════╤════════╤═══════╤═════════╕
│ name     │   height │   width │   grayscale │      R │      G │     B │   ALPHA │
╞══════════╪══════════╪═════════╪═════════════╪════════╪════════╪═══════╪═════════╡
│ bw       │      512 │     512 │       21.48 │   0.00 │   0.00 │  0.00 │    0.00 │
├──────────┼──────────┼─────────┼─────────────┼────────┼────────┼───────┼─────────┤
│ daffodil │      500 │     335 │        0.00 │ 109.25 │  85.56 │  4.97 │    0.00 │
├──────────┼──────────┼─────────┼─────────────┼────────┼────────┼───────┼─────────┤
│ eclipse  │      256 │     256 │        0.00 │ 109.05 │ 109.52 │ 39.85 │  133.59 │
├──────────┼──────────┼─────────┼─────────────┼────────┼────────┼───────┼─────────┤
│ trump    │      183 │     275 │        0.00 │  97.01 │  98.99 │ 90.92 │    0.00 │
╘══════════╧══════════╧═════════╧═════════════╧════════╧════════╧═══════╧═════════╛


Anche in questo caso combacia perfettamente come richiesto dalle specifiche di progetto.

# Note sullo Step 3

## Come funziona
Nello Step 3 l'obiettivo è esplorare le immagini già organizzate nella cartella files/images e ricavare, per ciascuna, dimensioni e statistiche di colore usando PIL e NumPy. Il flusso comincia aprendo con PIL ogni file immagine e trasformandolo in un array NumPy con np.array(img). A questo punto esamino la proprietà ndim: se l’array è bidimensionale (ndim == 2) significa che l’immagine è in scala di grigio, mentre un array tridimensionale (ndim == 3) con terza dimensione di lunghezza 3 indica un’immagine RGB, e di lunghezza 4 una RGBA dotata di canale alpha.

Una volta capito il formato, si ricavano altezza e larghezza dagli elementi di arr.shape. Per il canale di scala di grigio basta chiamare arr.mean() per ottenere la media dei pixel, mentre per RGB e RGBA ho sfruttato arr.mean(axis=(0,1)) per restituire in un solo passaggio i valori medi di R, G, B (e A, se presente). Man mano che le immagini vengono processate,  costruisco una riga di dati composta dal nome del file, dimensioni, media di grigio e medie dei canali, e la aggiungiamo a una collezione di risultati.

Infine, con la libreria tabulate ho convertito questa collezione in una tabella bella e leggibile: ogni riga corrisponde a un’immagine, con colonne per nome, altezza, larghezza, “grayscale” e poi R, G, B e ALPHA. Il risultato è una griglia compatta che ci permette di vedere subito non solo le dimensioni, ma anche un’idea del contenuto cromatico di ogni immagine.

# Grazie! 

Visita il mio [blog personale](https://stefanonocco.it/) o colleghiamoci su:  
- [LinkedIn](https://www.linkedin.com/in/stefanonocco/)  
- [Instagram](https://www.instagram.com/steno.py/)  

***Stefano Nocco***
